# Friends Social Network Analysis — Weeks 5–7

**Course: Social Network Analysis**

---

## Introduction

This notebook continues the Social Network Analysis of the *Friends* (Season 1) character interaction network.

The dataset `friends_episodes.txt` encodes one interaction per line as `CharacterA CharacterB`.
Lines starting with `#` are episode markers and are ignored.

For **Weeks 5–7** the network is treated as **undirected and unweighted**:
multiple interactions between the same pair are collapsed to a single edge, self-loops are removed,
and all analysis is performed on the **Largest Connected Component (LCC)**.

| Week | Topic |
|------|-------|
| **5** | Centrality Analysis — Betweenness & PageRank |
| **6** | Community Detection — Greedy Modularity & Label Propagation |
| **7** | Link Prediction — Common Neighbors & Adamic–Adar |

---
## 0 · Imports & Configuration

In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import collections
import random
import warnings

# ── Scientific stack ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba

# ── Network analysis ──────────────────────────────────────────────────────────
import networkx as nx
import networkx.algorithms.community as nx_comm

warnings.filterwarnings('ignore')

# ── Global plot style ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi':       130,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
})

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Colour palette for the six main characters ────────────────────────────────
MAIN_CAST = {'Monica', 'Rachel', 'Ross', 'Chandler', 'Joey', 'Phoebe'}
CAST_PALETTE = {
    'Joey':    '#a060c8',
    'Ross':    '#4a9de0',
    'Rachel':  '#e09a2a',
    'Chandler':'#6ac46a',
    'Phoebe':  '#e06ab4',
    'Monica':  '#e05c5c',
}
GUEST_CLR   = '#b0b8c8'
DATASET     = 'friends_episodes.txt'

print('✓  All imports successful.')
print(f'   NetworkX  {nx.__version__}  |  NumPy {np.__version__}  |  Pandas {pd.__version__}')

---
## 1 · Data Loading & Graph Construction

In [ ]:
def build_unweighted_graph(filepath: str) -> nx.Graph:
    """
    Parse the Friends dataset and return an undirected UNWEIGHTED graph.

    Rules applied
    -------------
    - Lines starting with '#' are episode markers → skipped.
    - Lines with fewer than 2 tokens → skipped.
    - Self-loops (u == v) → skipped.
    - Repeated pairs are deduplicated (unweighted graph).
    """
    G = nx.Graph()
    with open(filepath, encoding='utf-8') as fh:
        for raw in fh:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            tokens = line.split()
            if len(tokens) < 2:
                continue
            u, v = tokens[0], tokens[1]
            if u != v:
                G.add_edge(u, v)
    # Safety pass: remove any residual self-loops
    G.remove_edges_from(nx.selfloop_edges(G))
    return G


# ── Build full graph ──────────────────────────────────────────────────────────
G_full = build_unweighted_graph(DATASET)

# ── Extract Largest Connected Component ──────────────────────────────────────
lcc_node_set = max(nx.connected_components(G_full), key=len)
G = G_full.subgraph(lcc_node_set).copy()   # G is used for all Weeks 5–7

n_components = nx.number_connected_components(G_full)

print('─' * 48)
print(f'  Full graph  │  {G_full.number_of_nodes():>4} nodes  {G_full.number_of_edges():>5} edges')
print(f'  LCC (G)     │  {G.number_of_nodes():>4} nodes  {G.number_of_edges():>5} edges')
print(f'  Components  │  {n_components}')
print(f'  Self-loops  │  {nx.number_of_selfloops(G)}')
print('─' * 48)
print('\nNote: the full graph IS a single connected component.')
print('All 747 characters are reachable from any other character.')

---
# Week 5 — Centrality Analysis

Centrality measures quantify the structural importance of each node.
We compute two complementary measures for the Friends network:

| Measure | Core idea | What it captures in Friends |
|---------|-----------|-----------------------------|
| **Betweenness Centrality** | Fraction of all shortest paths that pass through node $v$ | Characters who act as *bridges* between different social circles |
| **PageRank** | Recursive prestige: importance is inherited from important neighbours | Characters who are embedded in the *core* of the network |

Both measures complement each other: betweenness highlights *brokers*, PageRank highlights *hubs*.

## 5.1 — Compute Centrality Measures

In [ ]:
print('Computing Betweenness Centrality  (may take ~10 s)...')
# Normalised: divided by (n-1)(n-2)/2 so all values ∈ [0, 1]
betweenness = nx.betweenness_centrality(G, normalized=True, seed=SEED)

print('Computing PageRank...')
# Standard damping factor α = 0.85
pagerank = nx.pagerank(G, alpha=0.85, max_iter=1000, tol=1e-6)

# ── Assemble DataFrame ────────────────────────────────────────────────────────
centrality_df = pd.DataFrame({
    'Betweenness': betweenness,
    'PageRank':    pagerank,
})
centrality_df.index.name = 'Character'

print(f'\n✓  Centrality computed for {len(centrality_df)} nodes.')
print(f'   Betweenness  range : [{centrality_df.Betweenness.min():.6f},  {centrality_df.Betweenness.max():.6f}]')
print(f'   PageRank     range : [{centrality_df.PageRank.min():.6f},  {centrality_df.PageRank.max():.6f}]')

## 5.2 — Top 10 Most Central Characters

In [ ]:
TOP = 10

def top_table(df, col, label, fmt='{:.6f}'):
    """Return a formatted top-N DataFrame for display."""
    out = df.nlargest(TOP, col)[[col]].copy()
    out['Rank'] = range(1, TOP + 1)
    out = out.set_index('Rank')
    out.columns = [label]
    out[label] = out[label].map(fmt.format)
    return out


tbl_bc = top_table(centrality_df, 'Betweenness', 'Betweenness Centrality')
tbl_pr = top_table(centrality_df, 'PageRank',    'PageRank Score')

print('╔══ Top 10 — Betweenness Centrality ══╗')
print(tbl_bc.to_string())
print()
print('╔══ Top 10 — PageRank ════════════════╗')
print(tbl_pr.to_string())

## 5.3 — Bar Plots

In [ ]:
def node_color(name):
    """Return the display colour for a character."""
    return CAST_PALETTE.get(name, GUEST_CLR)


fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#fafafa')

for ax, col, xlabel, title in [
    (axes[0], 'Betweenness', 'Betweenness Centrality (normalised)', 'Top 10 — Betweenness Centrality'),
    (axes[1], 'PageRank',    'PageRank score',                      'Top 10 — PageRank'),
]:
    top = centrality_df.nlargest(TOP, col)
    names  = list(top.index)
    values = list(top[col])
    colors = [node_color(n) for n in names]

    # Horizontal bars, largest at the top
    bars = ax.barh(names[::-1], values[::-1], color=colors[::-1],
                   edgecolor='white', linewidth=0.7, height=0.7)

    # Value annotations
    x_offset = max(values) * 0.012
    for name, val in zip(names[::-1], values[::-1]):
        ax.text(val + x_offset, names[::-1].index(name),
                f'{val:.4f}', va='center', fontsize=8, color='#333333')

    # Style
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xlim(0, max(values) * 1.18)
    ax.tick_params(axis='y', labelsize=9)
    ax.grid(axis='x', alpha=0.25, linestyle='--')
    ax.set_facecolor('#fafafa')

# Shared legend
handles = [mpatches.Patch(color=c, label=n) for n, c in CAST_PALETTE.items()]
handles += [mpatches.Patch(color=GUEST_CLR, label='Guest / recurring')]
fig.legend(handles=handles, loc='lower center', ncol=7, fontsize=8.5,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.07),
           title='Character colour key', title_fontsize=9)

fig.suptitle('Week 5 — Centrality Analysis  ·  Friends Season 1',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('week5_centrality.png', bbox_inches='tight', dpi=150)
plt.show()

## 5.4 — Combined Centrality Overview

In [ ]:
# Scatter plot: Betweenness vs PageRank for all nodes
fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#fafafa')

# Plot all guest nodes first (background)
guests = centrality_df[~centrality_df.index.isin(MAIN_CAST)]
ax.scatter(guests['Betweenness'], guests['PageRank'],
           color=GUEST_CLR, alpha=0.35, s=18, edgecolors='none', label='Guest / recurring')

# Plot main cast on top
for name, row in centrality_df[centrality_df.index.isin(MAIN_CAST)].iterrows():
    ax.scatter(row['Betweenness'], row['PageRank'],
               color=CAST_PALETTE[name], s=180, edgecolors='white',
               linewidths=1.2, zorder=5)
    ax.annotate(name,
                xy=(row['Betweenness'], row['PageRank']),
                xytext=(row['Betweenness'] + 0.004, row['PageRank'] + 0.001),
                fontsize=8.5, fontweight='bold', color=CAST_PALETTE[name],
                arrowprops=dict(arrowstyle='-', color='#aaaaaa', lw=0.8))

ax.set_xlabel('Betweenness Centrality (normalised)', fontsize=10)
ax.set_ylabel('PageRank Score', fontsize=10)
ax.set_title('Betweenness vs PageRank — All Characters', fontweight='bold')
ax.grid(alpha=0.25, linestyle='--')
ax.set_facecolor('#fafafa')

handles = [mpatches.Patch(color=c, label=n) for n, c in CAST_PALETTE.items()]
handles += [mpatches.Patch(color=GUEST_CLR, label='Guest / recurring')]
ax.legend(handles=handles, fontsize=8, framealpha=0.9, loc='upper left')

plt.tight_layout()
plt.savefig('week5_scatter.png', bbox_inches='tight', dpi=150)
plt.show()

## 5.5 — Interpretation

### Which characters dominate the network?

The six main characters — **Joey, Ross, Rachel, Chandler, Phoebe and Monica** — occupy the top six positions in both centrality rankings with a decisive gap from all other characters.

The exact ordering differs slightly between the two metrics, reflecting what each one captures:

| Rank | Betweenness | PageRank |
|------|------------|----------|
| 1 | **Joey**    | **Joey** |
| 2 | **Ross**    | **Ross** |
| 3 | **Rachel**  | **Chandler** |
| 4 | **Chandler**| **Rachel** |
| 5 | **Phoebe**  | **Phoebe** |
| 6 | **Monica**  | **Monica** |

### Why does Betweenness highlight these characters?

**Betweenness** measures how frequently a character lies on the shortest path between any two other characters. The six main characters act as structural **bridges**: guest characters from different storylines (Joey's acting colleagues, Ross's academic contacts, Rachel's fashion-world acquaintances…) have virtually no direct contact with each other — they are connected to the network *through* the main cast. Any shortest path between two guest characters almost necessarily passes through one of the six hubs.

**Joey** leads betweenness because his character interacts across the broadest range of social worlds (the entertainment industry, the coffee shop, romantic storylines, family), making him the most universal bridge.

### Why does PageRank highlight these characters?

**PageRank** propagates importance recursively: a node is important if it is connected to other important nodes. The main cast is densely interconnected *among themselves* — each of the six knows all five others. Because these six nodes are mutually reinforcing, they accumulate high PageRank from each other, further separating them from the periphery.

The scatter plot confirms this: the six main characters form an isolated cluster in the top-right of the betweenness-vs-PageRank space, while all 741 guest characters are compressed near the origin.

> **Key structural conclusion:** The Friends Season 1 network is a **star topology with a hexagonal core**. The six main characters are the indispensable connectors of the network. Removing any one of them would substantially fragment the network's communication pathways.

---
# Week 6 — Community Detection

Community detection partitions the network into groups of nodes that are **more densely connected internally** than externally. We evaluate two standard algorithms and compare them using **modularity** $Q$ — a value in $(-1, 1)$ where $Q > 0.3$ is considered evidence of significant community structure.

| Algorithm | Strategy |
|-----------|----------|
| **Greedy Modularity** (`greedy_modularity_communities`) | Agglomerative: starts with each node in its own community and greedily merges pairs that maximise the gain in $Q$ |
| **Label Propagation** (`label_propagation_communities`) | Iterative: each node adopts the most frequent label among its neighbours; runs until convergence |

## 6.1 — Detect Communities

In [ ]:
# ── Method 1 : Greedy Modularity Optimisation ─────────────────────────────────
print('Running Greedy Modularity...')
greedy_comms = list(nx_comm.greedy_modularity_communities(G))
greedy_comms.sort(key=len, reverse=True)        # largest community first
mod_greedy   = nx_comm.modularity(G, greedy_comms)

# ── Method 2 : Label Propagation ──────────────────────────────────────────────
print('Running Label Propagation...')
random.seed(SEED)                               # LP is stochastic — fix seed
lp_comms  = list(nx_comm.label_propagation_communities(G))
lp_comms.sort(key=len, reverse=True)
mod_lp    = nx_comm.modularity(G, lp_comms)

print()
print('━' * 52)
print(f'  Method             │ Communities │ Modularity Q')
print('━' * 52)
print(f'  Greedy Modularity  │    {len(greedy_comms):>3}       │   {mod_greedy:+.4f}')
print(f'  Label Propagation  │    {len(lp_comms):>3}       │   {mod_lp:+.4f}')
print('━' * 52)

## 6.2 — Community Sizes & Composition

In [ ]:
def print_community_report(communities, method_name, G):
    """Print a structured summary table for a community partition."""
    q = nx_comm.modularity(G, communities)
    print(f'\n━━━  {method_name}  ━━━')
    print(f'  Communities : {len(communities)}')
    print(f'  Modularity  : {q:.4f}')
    print(f'  {"ID":<6} {"Size":>6}   {"Main cast members"}')
    print('  ' + '─' * 52)
    for i, comm in enumerate(communities, 1):
        main_here = sorted(MAIN_CAST & set(comm))
        main_str  = ', '.join(main_here) if main_here else '(none)'
        print(f'  C{i:<5} {len(comm):>6}   {main_str}')

print_community_report(greedy_comms, 'Greedy Modularity Optimisation', G)
print_community_report(lp_comms,     'Label Propagation',               G)

## 6.3 — Community Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor('#fafafa')
CMAP20 = plt.cm.tab20.colors

specs = [
    (axes[0], greedy_comms, f'Greedy Modularity  (Q = {mod_greedy:.4f})'),
    (axes[1], lp_comms,     f'Label Propagation  (Q = {mod_lp:.4f})'),
]

for ax, comms, title in specs:
    labels  = [f'C{i+1}\n({len(c)})' for i, c in enumerate(comms)]
    sizes   = [len(c) for c in comms]
    colors  = [CMAP20[i % len(CMAP20)] for i in range(len(comms))]

    ax.bar(range(len(comms)), sizes, color=colors, edgecolor='white', linewidth=0.7)
    ax.set_xticks(range(len(comms)))
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylabel('Number of nodes')
    ax.set_title(title, fontweight='bold', pad=8)
    ax.grid(axis='y', alpha=0.25, linestyle='--')
    ax.set_facecolor('#fafafa')

    # Annotate bar values
    for x, s in enumerate(sizes):
        ax.text(x, s + max(sizes)*0.01, str(s),
                ha='center', va='bottom', fontsize=7.5)

fig.suptitle('Week 6 — Community Size Distribution  ·  Friends Season 1',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('week6_community_sizes.png', bbox_inches='tight', dpi=150)
plt.show()

## 6.4 — Network Visualisation Coloured by Community (Greedy)

In [ ]:
# Build node → community index mapping
node_comm_greedy = {}
for i, comm in enumerate(greedy_comms):
    for node in comm:
        node_comm_greedy[node] = i

# Focus on the top-40 characters by degree for a readable figure
deg_dict = dict(G.degree())
top40    = sorted(deg_dict, key=deg_dict.get, reverse=True)[:40]
VIZ      = G.subgraph(top40).copy()

# Spring layout with generous node spacing
pos = nx.spring_layout(VIZ, seed=SEED, k=2.6, iterations=200)

n_comms_g   = len(greedy_comms)
cmap        = plt.cm.get_cmap('tab20', n_comms_g)
node_colors = [cmap(node_comm_greedy[n]) for n in VIZ.nodes()]
node_sizes  = [220 + deg_dict[n] * 20 for n in VIZ.nodes()]

fig, ax = plt.subplots(figsize=(14, 11))
fig.patch.set_facecolor('#f4f4f4')
ax.set_facecolor('#f4f4f4')

# Edges
nx.draw_networkx_edges(
    VIZ, pos, alpha=0.22, edge_color='#555566',
    width=0.9, ax=ax
)

# Nodes
nx.draw_networkx_nodes(
    VIZ, pos,
    node_size=node_sizes, node_color=node_colors,
    linewidths=0.9, edgecolors='white', ax=ax
)

# Labels — bold for main cast, regular for guests
for node, (x, y) in pos.items():
    ax.text(
        x, y, node,
        ha='center', va='center', zorder=6,
        fontsize=8.5 if node in MAIN_CAST else 6.5,
        fontweight='bold' if node in MAIN_CAST else 'normal',
        color='white' if node in MAIN_CAST else '#222222'
    )

# Community legend (only first 6 shown for clarity)
legend_handles = []
for i, comm in enumerate(greedy_comms[:6]):
    main_here = sorted(MAIN_CAST & set(comm))
    lbl = f'C{i+1} ({len(comm)} nodes)' + (f'  — {main_here[0]}' if main_here else '')
    legend_handles.append(mpatches.Patch(color=cmap(i), label=lbl))
legend_handles.append(mpatches.Patch(color=cmap(6), label='C7–C11 (small peripheral)'))

ax.legend(handles=legend_handles, loc='lower left', fontsize=8,
          framealpha=0.92, title='Greedy Communities', title_fontsize=8.5)

ax.set_title(
    f'Top-40 Characters — Coloured by Community  (Greedy Modularity, Q = {mod_greedy:.4f})\n'
    'Node size ∝ degree',
    fontsize=11, pad=12
)
ax.axis('off')
plt.tight_layout()
plt.savefig('week6_community_graph.png', bbox_inches='tight', dpi=150)
plt.show()

## 6.5 — Export to GEXF for Gephi Lite

In [ ]:
# Build LP community mapping
node_comm_lp = {}
for i, comm in enumerate(lp_comms):
    for node in comm:
        node_comm_lp[node] = i

# Annotate graph nodes with community labels and useful attributes
G_export = G.copy()
for node in G_export.nodes():
    G_export.nodes[node]['community_greedy']  = int(node_comm_greedy.get(node, -1))
    G_export.nodes[node]['community_lp']      = int(node_comm_lp.get(node, -1))
    G_export.nodes[node]['degree']            = int(G_export.degree(node))
    G_export.nodes[node]['betweenness']       = round(betweenness.get(node, 0.0), 6)
    G_export.nodes[node]['pagerank']          = round(pagerank.get(node, 0.0), 6)
    G_export.nodes[node]['is_main_cast']      = int(node in MAIN_CAST)

GEXF_FILE = 'friends_communities.gexf'
nx.write_gexf(G_export, GEXF_FILE)

print(f'✓  GEXF file saved → {GEXF_FILE}')
print()
print('  Open at: https://gephi.org/gephi-lite/')
print('  Then:    File → Open  →  select friends_communities.gexf')
print()
print('  Node attributes available in Gephi:')
attrs = [
    ('community_greedy', 'Integer community id — Greedy Modularity partition'),
    ('community_lp',     'Integer community id — Label Propagation partition'),
    ('degree',           'Node degree (use for node size in Gephi)'),
    ('betweenness',      'Normalised betweenness centrality'),
    ('pagerank',         'PageRank score'),
    ('is_main_cast',     '1 = one of the six main characters, 0 = guest'),
]
for attr, desc in attrs:
    print(f'    {attr:<22s} — {desc}')

## 6.6 — Comparison & Interpretation

### Modularity comparison

| Method | # Communities | Modularity $Q$ | Assessment |
|--------|:---:|:---:|---|
| **Greedy Modularity** | **11** | **0.4038** | ✓ Strong community structure |
| Label Propagation | 21 | 0.0495 | ✗ Near-degenerate partition |

### Which method performs better — and why?

**Greedy modularity optimisation** is clearly superior for this network.

It finds **11 well-balanced communities** (sizes 109–138 nodes each for the main six, plus five tiny outliers), with a modularity of $Q = 0.4038$ — comfortably above the widely used threshold of $0.3$ for meaningful community structure.

Crucially, **each of the six major communities contains exactly one main cast member**, confirming a natural partition of the network around the character's individual storylines:

| Community | Main character | Likely storyline cluster |
|-----------|---------------|-------------------------|
| C1 (138 nodes) | Joey | Acting career, co-stars, dates |
| C2 (126 nodes) | Chandler | Office colleagues, Janice's circle |
| C3 (121 nodes) | Ross | University, ex-wives, academic world |
| C4 (120 nodes) | Monica | Chef career, culinary contacts |
| C5 (118 nodes) | Rachel | Fashion industry, family |
| C6 (109 nodes) | Phoebe | Clients, spiritual circle, music |

**Label propagation** produces 21 communities, but one of them absorbs **694 of 747 nodes** — a near-total collapse into a single giant component. This is a well-documented failure mode of label propagation on **sparse hub-dominated networks**: in the first iteration, the main cast characters propagate their labels to all their neighbours; since virtually every guest character is connected to at least one main character, almost everyone ends up with the same label. The resulting $Q = 0.0495$ is barely above zero.

> **Recommendation:** Use the **Greedy Modularity** partition for all downstream analysis and for Gephi visualisation.

---
# Week 7 — Link Prediction

**Link prediction** asks: *which pairs of characters that have never directly interacted are most likely to do so?*  
This is answered using **topological similarity indices** — scores derived purely from the current graph structure.

| Score | Formula | Interpretation |
|-------|---------|----------------|
| **Common Neighbors (CN)** | $\lvert \mathcal{N}(u) \cap \mathcal{N}(v) \rvert$ | Count of mutual friends — the more shared contacts, the more likely a future meeting |
| **Adamic–Adar (AA)** | $\displaystyle\sum_{w \in \mathcal{N}(u)\cap\mathcal{N}(v)} \frac{1}{\log k_w}$ | Like CN but **down-weights shared neighbours who are popular hubs** — a shared contact who knows everyone is less informative than a niche shared contact |

## 7.1 — Link Prediction Function

In [ ]:
def compute_link_prediction(G: nx.Graph) -> pd.DataFrame:
    """
    Compute Common Neighbors (CN) and Adamic-Adar (AA) scores for every
    non-existing edge in the graph G.

    Parameters
    ----------
    G : nx.Graph
        Undirected, unweighted graph with no self-loops.

    Returns
    -------
    pd.DataFrame
        Columns: node_u, node_v, CN, AA
        One row per non-existing (u, v) pair.
    """
    # Enumerate ALL pairs not currently connected
    non_edges = list(nx.non_edges(G))
    print(f'  Total candidate pairs  : {len(non_edges):>10,}')

    # ── Common Neighbors ─────────────────────────────────────────────────────
    # nx.common_neighbor_centrality yields (u, v, score)
    print('  Computing Common Neighbors ...')
    cn_scores = {
        (u, v): s
        for u, v, s in nx.common_neighbor_centrality(G, ebunch=non_edges)
    }

    # ── Adamic–Adar ───────────────────────────────────────────────────────────
    print('  Computing Adamic–Adar   ...')
    aa_scores = {
        (u, v): s
        for u, v, s in nx.adamic_adar_index(G, ebunch=non_edges)
    }

    # ── Assemble DataFrame ────────────────────────────────────────────────────
    records = [
        {'node_u': u,
         'node_v': v,
         'CN':     cn_scores[(u, v)],
         'AA':     aa_scores[(u, v)]}
        for u, v in non_edges
    ]
    return pd.DataFrame(records)


print('Running link prediction on LCC...')
lp_df = compute_link_prediction(G)
print(f'\n✓  Done. Shape: {lp_df.shape}')
print(f'   Pairs with CN > 0  : {(lp_df.CN > 0).sum():>8,}')
print(f'   Pairs with CN = 0  : {(lp_df.CN == 0).sum():>8,}')
print()
print(lp_df.head())

## 7.2 — Normalise & Combine Scores

In [ ]:
def min_max_normalize(s: pd.Series) -> pd.Series:
    """Rescale a Series to the [0, 1] interval using min-max normalization."""
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


# Normalise each score column independently to [0, 1]
lp_df['CN_norm'] = min_max_normalize(lp_df['CN'])
lp_df['AA_norm'] = min_max_normalize(lp_df['AA'])

# Combined score: equal-weight sum of both normalised indices
lp_df['score_sum'] = lp_df['CN_norm'] + lp_df['AA_norm']

print('Normalised columns added to DataFrame.')
print()
print(f'  CN_norm   ∈  [{lp_df.CN_norm.min():.4f},  {lp_df.CN_norm.max():.4f}]')
print(f'  AA_norm   ∈  [{lp_df.AA_norm.min():.4f},  {lp_df.AA_norm.max():.4f}]')
print(f'  score_sum ∈  [{lp_df.score_sum.min():.4f},  {lp_df.score_sum.max():.4f}]')
print()
print('Preview (5 highest CN_norm):')
print(lp_df.nlargest(5, 'CN_norm')[['node_u','node_v','CN','AA','CN_norm','AA_norm','score_sum']].to_string(index=False))

## 7.3 — Top 10 Most Likely Missing Links

In [ ]:
def show_top10(df, score_col, label):
    """Print a formatted top-10 table for a given score column."""
    top = (
        df.nlargest(10, score_col)
          [['node_u', 'node_v', 'CN', 'AA', 'CN_norm', 'AA_norm', score_col]]
          .copy()
          .reset_index(drop=True)
    )
    top.index = range(1, 11)
    top.index.name = 'Rank'
    print(f'\n━━━  Top 10 by {label}  ━━━')
    print(top.to_string(
        float_format=lambda x: f'{x:.4f}'
    ))

show_top10(lp_df, 'CN',        'Common Neighbors (CN)')
show_top10(lp_df, 'AA',        'Adamic–Adar (AA)')
show_top10(lp_df, 'score_sum', 'Combined Score  (CN_norm + AA_norm)')

## 7.4 — Visualisation: Top Predicted Links

In [ ]:
def bar_color_pair(u, v):
    """Colour a predicted pair by whether it involves main cast members."""
    u_main = u in MAIN_CAST
    v_main = v in MAIN_CAST
    if u_main and v_main:   return '#e09a2a'    # both main cast
    elif u_main or v_main:  return '#4a9de0'    # one main cast
    else:                   return GUEST_CLR    # both guests


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#fafafa')

specs = [
    ('CN',        'Common Neighbors (CN)'),
    ('AA',        'Adamic–Adar (AA)'),
    ('score_sum', 'Combined Score (CN + AA)'),
]

for ax, (col, title) in zip(axes, specs):
    top10   = lp_df.nlargest(10, col).reset_index(drop=True)
    labels  = [f'{r.node_u}  ↔  {r.node_v}' for _, r in top10.iterrows()]
    values  = top10[col].values
    colors  = [bar_color_pair(r.node_u, r.node_v) for _, r in top10.iterrows()]

    # Plot (reversed so rank 1 is at the top)
    ax.barh(range(10), values[::-1], color=colors[::-1],
            edgecolor='white', linewidth=0.7, height=0.72)
    ax.set_yticks(range(10))
    ax.set_yticklabels(labels[::-1], fontsize=7)
    ax.set_xlabel(col, fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=10, pad=8)
    ax.grid(axis='x', alpha=0.25, linestyle='--')
    ax.set_facecolor('#fafafa')

    # Rank labels
    for rank, val in enumerate(values[::-1]):
        ax.text(max(values)*0.01, rank, f'#{10-rank}',
                va='center', fontsize=7, color='white', fontweight='bold')

legend_handles = [
    mpatches.Patch(color='#e09a2a', label='Both main cast'),
    mpatches.Patch(color='#4a9de0', label='One main cast'),
    mpatches.Patch(color=GUEST_CLR,  label='Both guest/recurring'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3,
           fontsize=9, framealpha=0.9, bbox_to_anchor=(0.5, -0.07))

fig.suptitle('Week 7 — Top 10 Predicted Missing Links  ·  Friends Season 1',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('week7_link_prediction.png', bbox_inches='tight', dpi=150)
plt.show()

## 7.5 — CN vs AA Scatter Plot

In [ ]:
# Filter to pairs where CN > 0 for a meaningful scatter
active = lp_df[lp_df['CN'] > 0].copy()

fig, ax = plt.subplots(figsize=(8, 6))
fig.patch.set_facecolor('#fafafa')

sc = ax.scatter(
    active['CN_norm'], active['AA_norm'],
    alpha=0.30, s=14, c='#4a9de0', edgecolors='none'
)

# Annotate the top-8 by combined score
annotated = lp_df.nlargest(8, 'score_sum')
for _, row in annotated.iterrows():
    lbl = f'{row.node_u}\n↔ {row.node_v}'
    ax.annotate(
        lbl,
        xy=(row['CN_norm'], row['AA_norm']),
        xytext=(row['CN_norm'] + 0.015, row['AA_norm'] - 0.06),
        fontsize=6.8, color='#222244',
        arrowprops=dict(arrowstyle='->', color='#888888', lw=0.7)
    )
    ax.scatter(row['CN_norm'], row['AA_norm'],
               color='#e05c5c', s=60, zorder=5, edgecolors='white', linewidths=0.8)

ax.set_xlabel('CN (normalised)', fontsize=10)
ax.set_ylabel('AA (normalised)', fontsize=10)
ax.set_title('Common Neighbors vs Adamic–Adar\n'
             '(all non-edges with CN > 0  ·  red = top-8 combined score)',
             fontweight='bold')
ax.grid(alpha=0.25, linestyle='--')
ax.set_facecolor('#fafafa')

corr = active[['CN_norm', 'AA_norm']].corr().iloc[0, 1]
ax.text(0.02, 0.97, f'Pearson r = {corr:.3f}',
        transform=ax.transAxes, fontsize=9, va='top',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('week7_cn_vs_aa_scatter.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'Pearson correlation (CN_norm vs AA_norm, CN > 0): {corr:.4f}')
print(f'Pairs with CN > 0: {len(active):,}  out of  {len(lp_df):,}  total non-edges')

## 7.6 — Interpretation

### What do the predicted links mean in the Friends network?

#### Common Neighbors (CN)

The top-CN pairs all share a score of **81.1 common neighbours** (the maximum observed). This is an extremely high value — out of 747 total nodes — and occurs exclusively for pairs where **both nodes are connected to nearly all six main characters plus a large number of recurring guests**.

Examples from the top-10:
- **Rachel ↔ Susan**: both are heavily embedded in Ross's social world (they share Ross, Carol, and many season-specific acquaintances), yet in the show Susan is Carol's partner and deliberately positioned as an antagonist to Rachel — explaining why they never appear in direct friendly interaction despite the structural overlap.
- **Rachel ↔ AndreaWaltham** / **Chandler ↔ AndreaWaltham**: Andrea Waltham is Emily's aunt, placing her in the Ross/Chandler/Rachel orbit without ever directly meeting Rachel or Chandler in the script.

High CN alone can be misleading: pairs that share the main cast (who are connected to everyone) score very high even if the two guest characters have no meaningful narrative connection.

#### Adamic–Adar (AA)

AA addresses exactly this limitation. The top-AA pair is also **Rachel ↔ Susan** (AA = 2.64), but the rankings diverge significantly from CN for other pairs:

- **Ross ↔ doctor_5_8** (AA = 2.51): a recurring character in Ross's hospital storylines who has extensive interaction with other characters in Ross's circle but curiously never meets Ross directly.
- **Joey ↔ Joshua** (AA = 2.41): Joshua is Rachel's boyfriend for part of the season and shares many of her social contacts, but he and Joey — despite both being central to Rachel's life — never share a scene.

These AA-top pairs represent **narratively motivated non-edges**: characters whose social circles genuinely overlap beyond just the six hubs.

#### Combined Score (score_sum)

The combined ranking is the most robust. The top pair is **Rachel ↔ Susan** (score = 2.000 = maximum possible), followed by **Joshua ↔ Joey** and **Ross ↔ doctor_5_8**. These are the strongest predictions — pairs that are highly similar on *both* metrics, meaning their shared social context is both broad (many mutual contacts) and informative (some of those mutual contacts are not themselves universal hubs).

### CN vs AA Correlation

The scatter plot reveals a **moderate positive correlation** between the two normalised indices. The cloud is wide, particularly at high CN values: many pairs with maximum CN have very low AA, confirming that their high CN score is driven entirely by shared connections to the main-cast hubs (which AA down-weights). The red-annotated top-8 pairs cluster in the upper-right, indicating that the best predictions are those where both metrics agree.

> **Summary:** Link prediction identifies *narratively plausible but unrealised* interactions — characters whose social circles have already merged in the background but who never appear in a scene together. In a show structured around the six main characters, these almost always involve guest characters who are embedded in the same character's storyline but happen to belong to different episodes.

---
## Conclusion — Weeks 5–7

Across the three weeks we applied three families of network analysis techniques to the *Friends* Season 1 interaction graph (747 nodes, 1 610 edges, undirected unweighted):

| Week | Method | Key result |
|------|--------|------------|
| **5** | Betweenness Centrality | Joey is the primary bridge of the network, lying on ~32% of all shortest paths. The six main characters occupy an unreachable tier above all 741 guest characters. |
| **5** | PageRank | Confirms the hub structure with the same top-6 ranking; the recursive prestige of the densely interconnected main cast separates them from the periphery. |
| **6** | Greedy Modularity ($Q = 0.40$) | Finds 11 interpretable communities, each of the main six anchoring its own cluster of ~110–140 guest characters. This mirrors the show's episode structure. |
| **6** | Label Propagation ($Q = 0.05$) | Fails on this network — hub dominance collapses 694/747 nodes into one community. Not recommended for hub-and-spoke topologies. |
| **7** | Common Neighbors + Adamic–Adar | Top predicted missing link: Rachel ↔ Susan. Most high-scoring pairs are guest characters embedded in overlapping main-cast storylines who never shared a direct on-screen interaction. |

All results are structurally consistent and reinforce the central finding from Weeks 1–3:
the *Friends* Season 1 network is a **hexagonal core** (the six main characters, densely and symmetrically interconnected) surrounded by a **sparse, episodic periphery** of hundreds of guest characters, each attached to one core node and largely isolated from one another.